In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
import pandas as pd
from tqdm import tqdm
from utils import combined_approaches as ca
from utils import global_strategies as gs
from utils import label_based_measures as lbm
from utils import local_single_attribute as lsa
from utils import similarity_structures as ss
from utils import top_down_data_structures as tdds
from utils import value_overlap as vo
from utils import evaluation as eval

# ESERCIZIO DATA INTEGRATION - TOP DOWN

Considerare:

In [5]:
path="http://dbgroup.ing.unimore.it/EBI/TopB/"

src_links = [
  path + 'S1_.csv',
  path + 'S2_.csv',
  path + 'S3_.csv'   ]

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

GMT_GS=pd.read_csv(path +'GoldStandard_.csv').astype(str)
GlobalSchema=pd.read_csv(path +'GlobalSchema_.csv').astype(str)
tdds.to_GMM(GMT_GS)

SOURCE,S1,S2,S3
GAT,,,
l_address_phone,[],[],[address_phone]
l_city,[],[city],[city]
l_location,[new_location],[location],[location]
l_name_type,[],[],[name_type]
r_descrizione,[],[descrizione],[]
r_indirizzo,[],[indirizzo],[]
r_nome,[name],[nome],[]
r_telefono,[],[telefono],[]


In [6]:
ValutazioneMatchTable = pd.DataFrame(columns=['MT', 'TP', 'FP', 'FN', 'P', 'R', 'F'])

In [25]:
# Data

def CalcoloMatchingTable(TableL:pd.DataFrame,TableR:pd.DataFrame):

        SimTableA = lbm.levenshtein_label_based_similarity(TableL, TableR)
        #SimTableB = lbm.jaro_label_based_similarity(TableL, TableR)
        #SimTableB = vo.value_overlap_sim(GlobalSchema, Sources[y])
        SimTableC= vo.value_overlap_simjoin_jaccard(TableL, TableR, 0.3)

        # combiner
        # SimTable = ca.avg_sim_table([SimTableA,SimTableB,SimTableC])
        SimTable = ca.min_sim_table([SimTableA,SimTableC])
        #SimTable = ca.avg_sim_table([SimTableC])

        # Weighted-sum
        # SimTable = ca.Weighted_sum([SimTableA,SimTableB,SimTableC], [.3,.4,.3] )


        # dalla tabella di similarità alle corrispondenze

        MatchTable= lsa.thresholding(SimTable, 0.3)

        #MatchTable = lsa.top_K(MatchTable,1,'A')
        #MatchTable = lsa.top_K(MatchTable,1, 'B')

         # global mapping
        MatchTable = gs.stable_marriage(MatchTable)
        #MatchTable = gs.simmetric_best_match(MatchTable)

        return MatchTable

In [26]:
def CalcoloGlobalMatchingTable(Sources, GlobalSchema:pd.DataFrame):
    GlobalMatchingTable = pd.DataFrame(columns=['GAT','SOURCE','LAT','SLAT','sim'])
    for y in tqdm(Sources.keys()):
        MatchTable = CalcoloMatchingTable(GlobalSchema, Sources[y])
        
        MatchTable.columns = ['GAT','LAT','sim']
        MatchTable['SOURCE'] = str(y)
        MatchTable['SLAT'] = MatchTable['SOURCE']+'_'+MatchTable['LAT']
        GlobalMatchingTable = GlobalMatchingTable.append(MatchTable, sort=False)

    return GlobalMatchingTable

## Discussione

Lo svolgimento/risposta consiste nella discussione strutturata nei seguenti punti

1. Analizzare il gold standard dato (GoldStandard), stabilire il tipo di matching tra  il GlobalSchema e i  local schemata


2. Considerare la funzione CalcoloMatchingTable data, fare sua valutazione  e analizzare gli eventuali FP e/o FN


3. Modificare la funzione CalcoloMatchingTable,  mantendendo traccia delle modifiche e commentando brevemente i cambiamenti effettuati discutendo i   risultati ottenuti



In [11]:
eval.AnalisiGlobalMatchTable(GMT=GMT_GS, Sources=SOURCES)

1) Le seguenti SOURCE di GMT non sono definite in Sources: []
2) I seguenti SLAT di GMT non sono definiti in Sources: []
3) GAT mappati da una sola SOURCE: ['l_address_phone', 'l_name_type', 'r_descrizione', 'r_indirizzo', 'r_telefono']
4) GAT mappati in più LAT (per ciascuna SOURCE):
5) LAT mappati in più GAT (per ciascuna SOURCE):


In [27]:
GMTcalcolata=CalcoloGlobalMatchingTable(SOURCES, GlobalSchema)
tdds.to_GMM(GMTcalcolata)

100%|██████████| 3/3 [00:01<00:00,  2.00it/s]


SOURCE,S1,S2,S3
GAT,,,
l_address_phone,[],[],[address_phone]
l_city,[],[city],[city]
l_location,[new_location],[location],[location]
l_name_type,[],[],[name_type]
r_descrizione,[],[descrizione],[]
r_indirizzo,[],[indirizzo],[]
r_nome,[name],[nome],[]
r_telefono,[],[telefono],[]


In [28]:
X=eval.Valuta(GMT_GS[['GAT', 'SLAT']],GMTcalcolata[['GAT', 'SLAT']])
X

,MT,TP,FP,FN,P,R,F
0,12,12,0,0,1.0,1.0,1.0


In [29]:
ValutazioneMatchTable = ValutazioneMatchTable.append(X).rename(index={0: "Label-Instance-Min-0.3-StableMarriage"})
ValutazioneMatchTable

,MT,TP,FP,FN,P,R,F
Label-Instance-Avg-0.4-Top1-NoGlobal11,10,7,3,5,0.7000,0.5833,0.6364
Label-Instance-Avg-0.4-StableMarriage,10,8,2,4,0.8000,0.6667,0.7273
Label-Instance-Avg-0.3-StableMarriage,12,10,2,2,0.8333,0.8333,0.8333
Label-Instance-Min-0.3-StableMarriage,12,12,0,0,1.0000,1.0000,1.0000
